# 09 · 规划与控制：把轨迹放进闭环

预测的是别人可能怎么走，规划的是 ego 应该怎么走，控制则要让车辆真的跟上规划轨迹。这里用一个简化的 kinematic point model 与 pure-pursuit 控制器，研究 lookahead、速度、噪声和闭环误差。

学习目标：

- 从 reference path 生成控制命令；
- 对比 open-loop path 与 closed-loop executed path；
- 观察 lookahead 对稳定性和转弯误差的影响；
- 理解真实 planner 还需要障碍物约束、动力学、舒适性和实时性。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

plt.rcParams['figure.figsize'] = (9, 4.5)
plt.rcParams['axes.grid'] = True
rng = np.random.default_rng(5)

dt = 0.05
steps = 180
reference_x = np.linspace(0, 72, steps)
progress = np.clip((reference_x - 18) / 22, 0, 1)
reference_y = 2.2 * (3 * progress ** 2 - 2 * progress ** 3) + 0.45 * np.sin(reference_x / 5)
reference = np.c_[reference_x, reference_y]
reference_heading = np.arctan2(np.gradient(reference_y), np.gradient(reference_x))



In [ ]:
def wrap_angle(angle):
    return (angle + np.pi) % (2 * np.pi) - np.pi

def simulate_controller(lookahead=2.5, speed=8.0, steering_noise=0.0, gain=1.0, seed=5):
    local = np.random.default_rng(seed)
    x, y, yaw = reference[0, 0], reference[0, 1], reference_heading[0]
    trajectory = []
    steering = []
    for _ in range(steps):
        distances = np.sqrt((reference[:, 0] - x) ** 2 + (reference[:, 1] - y) ** 2)
        candidates = np.where(reference[:, 0] >= x + lookahead)[0]
        target_index = candidates[0] if len(candidates) else len(reference) - 1
        target = reference[target_index]
        target_angle = np.arctan2(target[1] - y, target[0] - x)
        heading_error = wrap_angle(target_angle - yaw)
        cross_track = np.sign(np.sin(yaw) * (target[0] - x) - np.cos(yaw) * (target[1] - y)) * distances[target_index]
        delta = gain * heading_error + 0.04 * cross_track
        delta += local.normal(0, steering_noise)
        delta = np.clip(delta, -0.55, 0.55)
        x += speed * np.cos(yaw) * dt
        y += speed * np.sin(yaw) * dt
        yaw = wrap_angle(yaw + speed * np.tan(delta) / 2.8 * dt)
        trajectory.append([x, y])
        steering.append(delta)
    return np.asarray(trajectory), np.asarray(steering)

trajectory, steering = simulate_controller()
print('closed-loop RMSE:', np.sqrt(np.mean((trajectory - reference[:len(trajectory)]) ** 2)))



In [ ]:
def show_control(lookahead=2.5, speed=8.0, steering_noise=0.0, gain=1.0):
    trajectory, steering = simulate_controller(
        lookahead=lookahead,
        speed=speed,
        steering_noise=steering_noise,
        gain=gain,
    )
    errors = np.linalg.norm(trajectory - reference[:len(trajectory)], axis=1)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(reference[:, 0], reference[:, 1], label='reference', linewidth=2)
    ax[0].plot(trajectory[:, 0], trajectory[:, 1], label='executed')
    ax[0].set_title(f'RMSE={np.sqrt(np.mean(errors ** 2)):.3f}, max={errors.max():.3f}')
    ax[0].set_xlabel('x / m')
    ax[0].set_ylabel('y / m')
    ax[0].legend()
    ax[1].plot(np.arange(len(steering)) * dt, steering)
    ax[1].set_title('steering command')
    ax[1].set_xlabel('time / s')
    ax[1].set_ylabel('steering / rad')
    plt.tight_layout()
    plt.show()

interact(
    show_control,
    lookahead=FloatSlider(min=0.5, max=8.0, step=0.25, value=2.5, description='lookahead'),
    speed=FloatSlider(min=3.0, max=14.0, step=0.5, value=8.0, description='speed'),
    steering_noise=FloatSlider(min=0.0, max=0.12, step=0.01, value=0.0, description='noise'),
    gain=FloatSlider(min=0.4, max=2.0, step=0.1, value=1.0, description='gain'),
);



### 练习：从 path tracking 到 planner contract

- 扫描 lookahead 和 speed，寻找误差突然增大的区域。
- 增加 steering rate limit，比较舒适性和跟踪误差。
- 在 reference path 上加入一个不可行障碍，设计一个最小风险 fallback。
- 把控制输出的单位、采样周期、最大曲率和最大 jerk 写成接口契约。


In [ ]:
lookaheads = np.linspace(0.75, 7.5, 12)
rmses = []
for value in lookaheads:
    path, _ = simulate_controller(lookahead=float(value))
    rmses.append(np.sqrt(np.mean((path - reference[:len(path)]) ** 2)))
plt.plot(lookaheads, rmses, marker='o')
plt.xlabel('lookahead / m')
plt.ylabel('closed-loop RMSE')
plt.title('lookahead ablation')
plt.show()



## 完成标准

完成后应能解释：

1. 为什么 open-loop 轨迹看起来平滑，不代表车辆能稳定跟踪？
2. lookahead、speed、steering noise 和 latency 如何共同影响闭环误差？
3. 一个可部署的 planner/controller 还需要哪些安全约束和监控指标？
4. 如何把这个 toy controller 接到 CARLA、nuPlan 或 NAVSIM 的评测接口？
